# 1. Getting started: build a plan once, evaluate many moment states

`cdfmm` is a fast multipole method (FMM) for the magnetic field of many
dipoles on **fixed geometry**. The defining workflow is:

```text
define geometry (positions, optional finite bodies)
    -> construct one reusable UniformFmm plan          (expensive, once)
    -> evaluate a moment state                         (cheap, repeated)
    -> update the moments, evaluate again ...
```

This tutorial walks through that lifecycle with point dipoles on the portable
CPU backend, compares the result with an exact direct sum, and shows what a
repeated evaluation costs. Units and array shapes are stated where they matter.

| Quantity | Symbol | Unit | Array |
|---|---|---|---|
| dipole positions (sources and targets) | $x_j$ | m | `(N, 3)` float64 |
| total dipole moment per source | $m_j = V_j M_j$ | A m$^2$ | `(N, 3)` float64 |
| magnetic field at a target | $H = -\nabla\phi$ | A/m | `(N, 3)` in the plan's precision |
| scalar potential (optional) | $\phi$ | A | `(N,)` |

The kernel is $G(r) = 1/(4\pi|r|)$, so a single dipole contributes
$H_{ij} = \frac{1}{4\pi}\left[\frac{3 r_{ij}(m_j\cdot r_{ij})}{|r_{ij}|^5} - \frac{m_j}{|r_{ij}|^3}\right]$
with $r_{ij} = x_i - x_j$. Inputs are **total moments**, never magnetisation:
the library performs no volume scaling.

In [ ]:
import time

import numpy as np

import cdfmm
from tutorial_utils import field_error_summary, quiet_construction

print(f"cdfmm module: {cdfmm.__file__}")
print(f"CUDA available: {cdfmm.cuda_full_available()}; oneMKL available: {cdfmm.one_mkl_available()}")

## A physical problem in SI units

$N$ dipoles fill a cube of side $L = 100\,$nm. Each represents a uniformly
magnetised micromagnetic cell of volume $V = L^3/N$ with saturation
magnetisation $M_s$ given through $\mu_0 M_s = 1.5\,$T, so the reference moment
is $m_{\rm ref} = M_s V$. Directions are isotropic and magnitudes vary between
0.5 and 1.5 $m_{\rm ref}$. A fixed seed makes the problem reproducible.

In [ ]:
N = 4000                      # dipoles
L_nm = 100.0                  # side of the simulation cube [nm]
mu0_Ms = 1.5                  # saturation magnetisation as mu0 * Ms [T]

mu0 = 4.0e-7 * np.pi          # [T m / A]
L = L_nm * 1.0e-9             # [m]
Ms = mu0_Ms / mu0             # [A/m]
cell_volume = L**3 / N        # [m^3]
moment_ref = Ms * cell_volume # [A m^2]

rng = np.random.default_rng(42)
positions = rng.uniform(-L / 2, L / 2, size=(N, 3))


def random_moments(rng):
    directions = rng.normal(size=(N, 3))
    directions /= np.linalg.norm(directions, axis=1)[:, None]
    scales = rng.uniform(0.5, 1.5, size=N)
    return directions * (moment_ref * scales)[:, None]


moments = random_moments(rng)
print(f"positions {positions.shape} {positions.dtype}; moments {moments.shape}")
print(f"cell size {cell_volume ** (1 / 3) * 1e9:.2f} nm; reference moment {moment_ref:.3e} A m^2")

## Options that define a plan

Everything in `UniformFmmOptions` is fixed at construction. The most important
choices:

| Option | Default | Meaning |
|---|---|---|
| `expansion_order` | 4 | truncation order $p$ of the multipole/local expansions; accuracy grows with $p$ |
| `tree.max_level` | 0 | depth of the complete octree; deeper trees move work from the near field (P2P) to the far field (M2L) |
| `precision` | `FLOAT32` | scalar of every stored operator, coefficient and result (`FLOAT64` available) |
| `expansion_basis` | `SPHERICAL` | real spherical harmonics, $(p+1)^2$ coefficients; `CARTESIAN` stores $(p+1)(p+2)(p+3)/6$ |
| `backend` | `AUTO` | `AUTO` resolves to the portable `CPU_STATIC`; CUDA and oneMKL are explicit choices (tutorial 3) |
| `fixed_target_source_indices` | none | immutable self-identity map when targets *are* the sources |
| `enable_cache` | `True` | persistent operator/geometry cache on disk (tutorial 4) |

A plan is `UniformFmm(source_positions, target_positions, options)`. Here the
targets are the sources themselves, so each target must **exclude its own
singular self pair**: that is what the identity map `target_source_indices`
does. Identity is by index, never by coordinate equality.

In [ ]:
identities = np.arange(N, dtype=np.int32)

options = cdfmm.UniformFmmOptions()
options.expansion_order = 6
options.tree.max_level = 3
options.precision = cdfmm.StaticPrecision.FLOAT32
options.expansion_basis = cdfmm.ExpansionBasis.SPHERICAL
options.backend = cdfmm.ExecutionBackend.AUTO
options.fixed_target_source_indices = identities.tolist()

start = time.perf_counter()
plan = cdfmm.UniformFmm(positions, positions, options)
setup_seconds = time.perf_counter() - start
print(f"\nplan built in {setup_seconds:.3f} s; backend {plan.backend}, precision {plan.precision}")

Every construction prints an **initialisation summary** from the C++ core: the
requested options, the resolved backend and executors, the resolved P2P
packing, the tree, the physical geometry and the cache state. The same
summary appears for C++, Python, C and Fortran callers. The tutorials silence
it inside loops with `quiet_construction()` from `tutorial_utils`; the
resolved choices are also available programmatically, for example
`plan.backend`, `plan.p2p_execution_packing` and `plan.static_plan_statistics`.

Construction did the geometry-dependent work: the octree, the Morton sort, the
interaction lists, the reusable P2M/M2M/M2L/L2L/L2P operators and the exact
near-field tensors. None of it depends on the moments.

## One evaluation and an exact reference

`evaluate` takes the moments **in the original source order**, permutes them
internally, runs the FMM and returns results **in the original target order**.
The direct $O(N^2)$ sum `direct_p2p_reference` is the exact reference.

In [ ]:
start = time.perf_counter()
result = plan.evaluate(moments, output="field", target_source_indices=identities)
evaluation_seconds = time.perf_counter() - start
H_fmm = result["H"]

start = time.perf_counter()
H_direct = cdfmm.direct_p2p_reference(
    positions, positions, moments, output="field",
    target_source_indices=identities,
)["H"]
direct_seconds = time.perf_counter() - start

errors = field_error_summary(H_fmm, H_direct)
print(f"H shape {H_fmm.shape}, dtype {H_fmm.dtype} (the plan's precision)")
print(f"FMM evaluation   {evaluation_seconds * 1e3:8.2f} ms")
print(f"direct reference {direct_seconds * 1e3:8.2f} ms")
print(f"relative L2 error {errors['relative_l2']:.2e}; max pointwise {errors['max_relative']:.2e}")

The error is the truncation error of the far-field expansion at $p = 6$ plus
FP32 rounding; the near field is exact. Raising `expansion_order` lowers it
(tutorial 5 shows how to choose $p$ and the depth systematically).

## Repeated evaluation: the reason the plan exists

Only the moments change between calls. Nothing is rebuilt, nothing is
re-uploaded, and the per-phase timings are available after every call.

In [ ]:
states = [random_moments(rng) for _ in range(5)]
# Internal phase timing is off by default (the production path); switch it on
# for this diagnostic loop.  Results and the resolved plan do not change.
plan.set_timing_level(cdfmm.TimingLevel.DETAILED)
timings = []
for state in states:
    start = time.perf_counter()
    H = plan.evaluate(state, output="field", target_source_indices=identities)["H"]
    timings.append(time.perf_counter() - start)
print("per-evaluation wall time [ms]:", ", ".join(f"{t * 1e3:.2f}" for t in timings))

phases = plan.last_timings
for name in ("moment_permutation", "p2m", "m2m", "m2l", "l2l", "l2p", "p2p",
             "result_unpermutation", "total"):
    print(f"  {name:22s} {phases[name] * 1e3:8.3f} ms")

## What triggers a rebuild

| Change | Action |
|---|---|
| moments | `plan.evaluate(new_moments, ...)` |
| self-identity map (when not fixed in the options) | pass a different `target_source_indices` |
| positions, finite-body records, order, depth, basis, precision, backend, periodic cell | construct a new `UniformFmm` |

The same holds for the persistent cache: a rebuilt plan with unchanged
geometry loads its operators from disk (tutorial 4).

## Optional: the same plan on CUDA

`CUDA_FULL` keeps operators, coefficients and the near-field data on the
device; a repeated evaluation uploads only the moments and downloads only the
field. The cell is skipped when the build has no CUDA device.

In [ ]:
if cdfmm.cuda_full_available():
    cuda_options = cdfmm.UniformFmmOptions()
    cuda_options.expansion_order = options.expansion_order
    cuda_options.tree.max_level = options.tree.max_level
    cuda_options.fixed_target_source_indices = identities.tolist()
    cuda_options.backend = cdfmm.ExecutionBackend.CUDA_FULL
    with quiet_construction():
        cuda_plan = cdfmm.UniformFmm(positions, positions, cuda_options)
    cuda_plan.evaluate(moments, target_source_indices=identities)   # warm up
    start = time.perf_counter()
    H_cuda = cuda_plan.evaluate(moments, target_source_indices=identities)["H"]
    cuda_seconds = time.perf_counter() - start
    cuda_errors = field_error_summary(H_cuda, H_direct)
    statistics = cuda_plan.cuda_plan_statistics
    print(f"CUDA_FULL evaluation {cuda_seconds * 1e3:.2f} ms; relative L2 {cuda_errors['relative_l2']:.2e}")
    print(f"per evaluation: {statistics['evaluation_h2d_bytes']} bytes up, "
          f"{statistics['evaluation_d2h_bytes']} bytes down; "
          f"{statistics['persistent_device_bytes'] / 2**20:.1f} MiB resident")
else:
    print("CUDA is not available in this build; the CPU plan above is the complete workflow.")

## Where next

- **Tutorial 2** replaces point dipoles by rectangular prisms and tetrahedra
  with exact finite-body near fields.
- **Tutorial 3** covers backends, precision, and the explicit execution
  options most users can leave on `AUTO`.
- **Tutorial 4** covers the persistent cache and periodic boundary conditions.
- **Tutorial 5** covers the tree, adaptive topologies and choosing $p$ and depth.
- **Tutorial 6** takes the FMM apart operator by operator.